# Chronic Kidney Disease (CKD) Screening System — Model Training & Evaluation

**Institution:** Dayananda Sagar College of Engineering, Bengaluru  
**Department:** Department of Artificial Intelligence and Machine Learning  
**Course:** BCP308 – Community Project  
**Team:** A-12 (Ayush Anjan Choudhary - Lead, Harsh Kumawat, Krishna Arora, Kunal Kumar Singh)  

This notebook trains and evaluates the machine learning models (Decision Tree and Random Forest) using the authentic **UCI Chronic Kidney Disease Dataset** and exports the production pipeline `ckd_model.pkl`.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

## 1. Load and Inspect Cleaned Dataset

In [2]:
dataset_path = os.path.join('..', 'ml', 'dataset', 'chronic_kidney_disease_cleaned.csv')
df = pd.read_csv(dataset_path)
print('Dataset Shape:', df.shape)
print('Class Distribution:\n', df['class'].value_counts())

## 2. Define Features & Preprocessing Pipeline

In [3]:
numerical_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
categorical_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
target_col = 'class'

X = df[numerical_cols + categorical_cols].copy()
y = df[target_col].map({'ckd': 1, 'notckd': 0}).astype(int)

# 80% Train, 20% Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

## 3. Train Decision Tree and Random Forest

In [4]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, min_samples_split=4, min_samples_leaf=2, class_weight='balanced', random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_split=4, min_samples_leaf=2, class_weight='balanced', random_state=42))
])

# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dt_cv = cross_validate(dt_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'recall', 'roc_auc'])
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'recall', 'roc_auc'])

print('Decision Tree CV Acc:', dt_cv['test_accuracy'].mean(), 'Recall:', dt_cv['test_recall'].mean())
print('Random Forest CV Acc:', rf_cv['test_accuracy'].mean(), 'Recall:', rf_cv['test_recall'].mean())

## 4. Evaluate on Held-Out Test Set (N=80)

In [5]:
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print('Random Forest Test Accuracy:', accuracy_score(y_test, y_pred))
print('Random Forest Sensitivity (Recall):', recall_score(y_test, y_pred))
print('Random Forest ROC-AUC:', roc_auc_score(y_test, y_prob))
print('Confusion Matrix:\n', cm)

## 5. Export Champion Pipeline to `ckd_model.pkl`

In [6]:
output_dir = os.path.join('..', 'ml', 'models')
os.makedirs(output_dir, exist_ok=True)
joblib.dump(rf_pipeline, os.path.join(output_dir, 'ckd_model.pkl'))
joblib.dump(rf_pipeline, os.path.join(output_dir, 'ckd_pipeline.joblib'))
print('Exported ckd_model.pkl successfully!')